# <div align="center"> Statistical Learning </div>
# <div align="center"> Machine Learning and Econometrics </div>
## <div align="center"> ECO 4971/6971 </div>
#### <div align="center">Class 12 — Hands On Neural Networks: Exercise Solutions</div>
<div align="center"> Jonathan Holmes (he/him)</div>

These exercises accompany the *Hands On: Neural Networks with Keras* lecture notebook, which fits a sequence of feedforward networks to the **Wage** dataset (predicting `log(wage)` from `age`, then from `age + education + jobclass`). Some answers refer to specific output values from running the notebook; with `random_state = 1706` the numbers should match closely, but small run-to-run variation is normal because TensorFlow training is not perfectly deterministic.

# In-Class Exercise #1 — Shallow network vs. linear baseline

**Q1. Look at the two plots side by side. How does the shallow network's fitted curve differ from the linear fit? In which parts of the age range is the added flexibility most apparent?**

---

**Answer.**

The **linear fit** is a single straight line: it captures the average upward slope of `log(wage)` with `age` but cannot bend. It systematically *under-predicts* in the middle of the age range (where wages peak) and *over-predicts* in the tails (very young and near-retirement workers).

The **shallow ReLU network** produces a piecewise-linear curve made up of as many segments as it has hidden units (each ReLU unit contributes one kink). With `N_UNITS = 64`, the curve smoothly traces the classic age–wage *arc*: it rises steeply in the 20s and 30s, levels off through the 40s and 50s, and bends back downward in the 60s.

The added flexibility is most apparent in three places:

1. **Under age ~25**: the rapid early rise — wages climb quickly with the first years of experience — which a single line cannot replicate.
2. **The mid-career peak around 45–55**: the line is monotonic, so it cannot show a peak; the network captures it with no polynomial terms required.
3. **Past age ~60**: the slight decline as workers approach retirement; the linear fit either misses this or is forced to compromise the slope earlier in the range.

The shallow network is doing exactly what we asked for in Class 9 with polynomial regression — fitting a curved relationship — but it learns the curvature *from the data* instead of us specifying $\text{age}^2$, $\text{age}^3$, etc. by hand.

**Q2. `EarlyStopping` halts training when the validation loss stops improving for `patience` consecutive epochs. Why do we monitor validation loss rather than training loss for this decision? What would happen if we only monitored training loss?**

---

**Answer.**

Training loss measures fit on the *same* data the optimizer is updating against. As long as the model has any capacity left, gradient descent can almost always lower training loss further by chasing finer and finer details — including the noise. So training loss is **monotonically non-increasing in expectation** and would never give early stopping a reason to halt. If we monitored training loss, the algorithm would essentially train forever (or until the maximum epoch limit), giving us a model that has memorized the training set.

Validation loss is computed on data the optimizer never sees during gradient updates. It tracks **out-of-sample performance** — the same thing we ultimately care about. Early in training, validation loss falls along with training loss because both reflect real signal being learned. Eventually, however, the optimizer starts fitting training-set noise; validation loss flattens and then starts to *rise* even as training loss keeps dropping. That divergence is the signature of overfitting, and `EarlyStopping(monitor='val_loss')` halts precisely at the validation minimum (with `restore_best_weights=True` rolling the network back to the weights at that minimum).

Short version: training loss tells us how well we have fit the past; validation loss is a proxy for how well we will predict the future. Early stopping should be driven by the latter.

# In-Class Exercise #2 — Reading training curves

**Q3. In the training curve above, do you see evidence of overfitting? How many epochs did early stopping allow before halting?**

---

**Answer.**

Look for the classic divergence pattern: training loss (blue) keeps falling smoothly, while validation loss (orange) flattens and may start ticking upward. A small persistent gap between training and validation loss is normal — it reflects the modest extra error of generalizing to unseen data — but a *widening* gap is overfitting.

On the Wage data with the default `N_UNITS = 64` and `patience = 30`, what you should see is:

* The losses fall together quickly for the first ~50 epochs.
* Validation loss bottoms out somewhere between epoch 100 and epoch 200, then plateaus or drifts upward.
* Early stopping halts roughly 30 epochs after that minimum (because of `patience = 30`) and restores the best weights, so the trained network is the one at the validation minimum, not at the stopping point.

The exact halting epoch will vary run-to-run (TensorFlow optimization is not perfectly deterministic), but a typical run halts somewhere in the **120–230 epoch** range. The signs of overfitting on a noisy dataset like Wage are usually mild rather than dramatic — the model is small enough that it cannot drive training loss to zero — but the validation curve almost always plateaus before the training curve does, which is enough for early stopping to do useful work.

**Q4. Re-run Part 3 with `N_UNITS = 256`. How do the training curves change compared to `N_UNITS = 64`? Does the test MSE improve, stay the same, or get worse?**

---

**Answer.**

With `N_UNITS = 256`, the network has roughly $4\times$ more hidden units and therefore much more capacity:

* **Training loss falls faster and reaches a lower floor** — more units means more flexibility to fit fine details of the training data.
* **Validation loss** typically falls almost as fast for the first few dozen epochs, then either:
  * plateaus at roughly the same level as the 64-unit model (the extra capacity is not buying real signal), or
  * starts to *rise sooner* than with 64 units (the extra capacity is fitting noise faster — more visible overfitting).
* **The gap between training and validation loss widens**, which is the textbook signature of higher variance.

**Test MSE on the Wage data with `age` as the only predictor usually changes very little** when going from 64 to 256 units. Why? Because the true age–wage relationship is essentially a smooth one-dimensional curve — there is not much for the extra units to learn that the smaller network was missing. The bias was already low; you are mostly buying variance.

This is a useful lesson: **bigger is not automatically better**. Doubling or quadrupling capacity helps when the model was *too simple* for the task, but past that point you are paying in variance for capacity you do not need. Cross-validation (or a validation curve) is the right way to decide which width actually generalizes best.

# In-Class Exercise #3 — Neural net vs. OLS with all features

**Q5. Look at the comparison table. Does the neural network with all features outperform OLS? Is the gap large in practical terms — would it meaningfully change a policy or business decision?**

---

**Answer.**

On the Wage dataset, the two-hidden-layer network with all features typically achieves a test MSE that is **slightly lower** than OLS — a few percent improvement in mean squared error. So *technically* the network wins. But in practical terms, the gap is usually small:

* The Wage dataset has $n = 3{,}000$ male workers and a handful of features. There is genuine nonlinearity (the age arc) and some interaction between age and education, but most of the signal is already linear or near-linear.
* OLS captures the dummy effects of `education` and `jobclass` perfectly, and the linear age slope captures most of the age variation. The neural network only adds curvature on top.
* Most of the residual variance is **irreducible** — wages depend on many things we have not measured (industry, region, ability, hours, bargaining), and no model can predict that part from these four features.

**Would it change a policy or business decision?** Almost certainly not. If a labour ministry uses these models to forecast median wages by demographic group, OLS and the neural net will give answers that differ by a few percent — well inside the standard error. The choice between them should be made on grounds *other* than this small accuracy gap: simplicity, interpretability, ease of communication, ability to compute counterfactuals.

This generalizes a useful piece of practical advice: **on small-to-medium tabular datasets with mostly linear signal, regularized linear models are very hard to beat by enough to matter**. Neural networks really shine when you have lots of data, complex high-dimensional inputs (images, text, audio), or strongly nonlinear interactions that linear models cannot represent.

**Q6. In Class 10 we used Lasso to control model complexity by shrinking coefficients toward zero. A neural network also controls complexity — through architecture choices (layers, units) and early stopping. What is the key difference in *how* each method selects which features to use?**

---

**Answer.**

**Lasso** does *explicit, hard variable selection*. Its $\ell_1$ penalty drives some coefficients **exactly to zero** at a sufficiently large $\lambda$, removing those predictors from the model entirely. The output is a literal list of features that survived the shrinkage and a list of features that did not. You can read the model and say: "Lasso kept Income, Limit, and Student; it dropped the rest." That sparsity is a defining feature of Lasso — it is a model-selection method as well as a regularizer.

**A neural network**, in contrast, almost never sets a feature's effective contribution to zero. Every input feeds into every hidden unit through some weight; even after early stopping or weight decay, those weights are *small but not exactly zero*. The network does not select features — it **distributes attention** across them, smoothly downweighting those that are uninformative and upweighting those that matter, often through nonlinear combinations and interactions. There is no clean list of "included" vs. "excluded" features; only weight matrices that you have to interpret indirectly (e.g., via permutation importance, gradient-based attributions, or SHAP values).

A practical consequence: if a stakeholder asks *"which variables matter for this prediction?"*, Lasso gives a one-line answer. A neural network requires extra interpretation tools to give an honest answer to the same question.

**Q7. A policymaker asks you to explain your wage model. Which would you show them — OLS or the neural network — and why? What does your answer depend on?**

---

**Answer.**

For a policymaker, **show OLS**. The reason is *not* that OLS is more accurate (it usually isn't, by a small margin), but that OLS has properties the neural network lacks:

* **Per-coefficient interpretability.** "Holding education and jobclass fixed, an extra year of age is associated with $X$% more in wages." That is a sentence a policymaker can act on.
* **Standard errors and confidence intervals** that they (or their analysts) already know how to read.
* **A direct path to counterfactuals** — "if we shifted the education distribution, what would happen to predicted average wages?" — using only the coefficient table.
* **Reproducibility and auditability**: anyone with the dataset can refit OLS in seconds and verify the result. A neural network's training depends on initialization, batching order, and stopping criteria; even with a fixed seed, exact reproduction across hardware can be tricky.

**What does the answer depend on?** Two things:

1. **Why the policymaker is asking.** If they want to understand the structure of the wage distribution and explain it to others, OLS wins. If they want the most accurate possible forecast for an automated system that will not be explained to anyone, the neural network's small accuracy edge may be worth the loss of transparency. Most government and regulatory contexts are firmly in the first category.
2. **Whether prediction or inference is the goal.** If the question is *"what is this person's expected wage?"*, both models can answer. If the question is *"what is the effect of one more year of education on wages?"*, neither model is causal on its own — but OLS at least makes the partial-association interpretation explicit and forces you to confront the assumption that the other regressors capture the relevant confounders. The neural network buries that assumption inside the architecture and is harder to scrutinize.

A defensible default: **use OLS for communication and the neural network as a benchmark or a sanity check**. If the network beats OLS by a meaningful margin, that is a signal to revisit OLS — perhaps add interactions, polynomial terms, or splines — rather than a reason to deploy a black-box model when an interpretable one will do.